In [9]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight

In [10]:
TRAIN_GROUNDTRUTH_PATH = "/Users/mac/Development/diploma/skin-mutation-detection/ISIC_2020_Dataset/ISIC_2020_Train_Metadata.csv"

TRAIN_IMAGE_DIR = "/Users/mac/Development/diploma/skin-mutation-detection/ISIC_2020_Dataset/train"

IMG_SIZE = (456, 456)
BATCH_SIZE = 32
EPOCHS = 20
LABELS = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']

def read_ds(gnd_path):
    train_groundtruth = pd.read_csv(gnd_path)
    return train_groundtruth

def is_recommended(x):
    return x['target']


def get_image_paths_and_labels(df: pd.DataFrame, image_dir):
    image_paths = [os.path.join(image_dir, row['image_name'] + '.jpg') for _, row in df.iterrows()]
    labels = df.target
    
    return image_paths, labels.tolist()


train_groundtruth = read_ds(TRAIN_GROUNDTRUTH_PATH)

train_paths, train_labels = get_image_paths_and_labels(train_groundtruth, TRAIN_IMAGE_DIR)


In [11]:
def load_image(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img, label

def preprocess_and_augment(file_path, label):
    img, label = load_image(file_path, label)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.2)
    img = tf.image.random_contrast(img, lower=0.8, upper=1.2)
    return img, label

# Define the fraction for validation and test splits
VAL_SPLIT = 0.2
TEST_SPLIT = 0.1

# Shuffle the dataset
dataset = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
dataset = dataset.shuffle(512)

# Separate the dataset into two groups based on label
label_0_paths = [train_paths[i] for i in range(len(train_labels)) if train_labels[i] == 0]
label_0_labels = [train_labels[i] for i in range(len(train_labels)) if train_labels[i] == 0]

label_1_paths = [train_paths[i] for i in range(len(train_labels)) if train_labels[i] == 1]
label_1_labels = [train_labels[i] for i in range(len(train_labels)) if train_labels[i] == 1]

# Shuffle each label group independently
label_0_dataset = tf.data.Dataset.from_tensor_slices((label_0_paths, label_0_labels)).shuffle(512)
label_1_dataset = tf.data.Dataset.from_tensor_slices((label_1_paths, label_1_labels)).shuffle(512)

# Calculate split sizes based on the dataset size
val_size_0 = int(len(label_0_paths) * VAL_SPLIT)
val_size_1 = int(len(label_1_paths) * VAL_SPLIT)
test_size_0 = int(len(label_0_paths) * TEST_SPLIT)
test_size_1 = int(len(label_1_paths) * TEST_SPLIT)

train_size_0 = len(label_0_paths) - val_size_0 - test_size_0
train_size_1 = len(label_1_paths) - val_size_1 - test_size_1

# Create train, validation, and test datasets for label 0
train_label_0 = label_0_dataset.take(train_size_0).map(preprocess_and_augment).batch(BATCH_SIZE).cache("isis-train.ds").prefetch(tf.data.AUTOTUNE)
val_label_0 = label_0_dataset.skip(train_size_0).take(val_size_0).map(load_image).batch(BATCH_SIZE).cache("isis-val.ds").prefetch(tf.data.AUTOTUNE)
test_label_0 = label_0_dataset.skip(train_size_0 + val_size_0).take(test_size_0).map(load_image).cache("isis-test.ds").batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Create train, validation, and test datasets for label 1
train_label_1 = label_1_dataset.take(train_size_1).map(preprocess_and_augment).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_label_1 = label_1_dataset.skip(train_size_1).take(val_size_1).map(load_image).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_label_1 = label_1_dataset.skip(train_size_1 + val_size_1).take(test_size_1).map(load_image).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Combine label 0 and label 1 datasets for training, validation, and test
train_dataset = train_label_0.concatenate(train_label_1).shuffle(128)
val_dataset = val_label_0.concatenate(val_label_1).shuffle(128)
test_dataset = test_label_0.concatenate(test_label_1).shuffle(128)

In [12]:
def build_model():
    base_model = EfficientNetV2S(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3))
    base_model.trainable = False
    
    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(2048, activation="mish")(x)  
    x = layers.Dropout(0.4)(x)  
    x = layers.LayerNormalization()(x)

    x = layers.Dense(1024, activation="mish")(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(512, activation="mish")(x)
    x = layers.Dropout(0.2)(x)
    x = layers.BatchNormalization()(x)
    
    outputs = layers.Dense(1, activation="sigmoid")(x)
    
    model = models.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), 
                  loss="binary_crossentropy", 
                  metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])
    return model

model = build_model()


KeyboardInterrupt: 

In [ ]:
train_labels = np.array(train_labels)  # Ensure train_labels is a NumPy array

unique_classes = np.unique(train_labels)  # Get unique class labels
class_weights = compute_class_weight(class_weight="balanced", classes=unique_classes, y=train_labels)

class_weights_dict = {i: weight for i, weight in zip(unique_classes, class_weights)}


In [ ]:
class_weights_dict

{np.int64(0): np.float64(0.5089730194825149),
 np.int64(1): np.float64(28.361301369863014)}

In [ ]:
checkpoint = ModelCheckpoint("efficientnet-v2-s-recommendation-isic20-deep_baserun.keras", monitor="val_accuracy", save_best_only=True, mode="max", verbose=1)
history = model.fit(train_dataset, validation_data=val_dataset, epochs=EPOCHS, callbacks=[checkpoint],
                    class_weight=class_weights_dict)

Epoch 1/20


2025-03-17 13:56:45.131967: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


725/725 ━━━━━━━━━━━━━━━━━━━━ 0s 792ms/step - accuracy: 0.6649 - loss: 0.4181 - precision: 0.0013 - recall: 0.0208
Epoch 1: val_accuracy improved from -inf to 0.96679, saving model to efficientnet-v2-s-recommendation-isic20-deep_baserun.keras
725/725 ━━━━━━━━━━━━━━━━━━━━ 761s 1s/step - accuracy: 0.6650 - loss: 0.4190 - precision: 0.0013 - recall: 0.0210 - val_accuracy: 0.9668 - val_loss: 0.3393 - val_precision: 0.1176 - val_recall: 0.1379
Epoch 2/20
725/725 ━━━━━━━━━━━━━━━━━━━━ 0s 807ms/step - accuracy: 0.8459 - loss: 0.2799 - precision: 0.0020 - recall: 0.0245
Epoch 2: val_accuracy did not improve from 0.96679
725/725 ━━━━━━━━━━━━━━━━━━━━ 747s 1s/step - accuracy: 0.8459 - loss: 0.2807 - precision: 0.0020 - recall: 0.0246 - val_accuracy: 0.9318 - val_loss: 0.5022 - val_precision: 0.0532 - val_recall: 0.1724
Epoch 3/20
725/725 ━━━━━━━━━━━━━━━━━━━━ 0s 853ms/step - accuracy: 0.8152 - loss: 0.2845 - precision: 0.0016 - recall: 0.0289
Epoch 3: val_accuracy did not improve from 0.96679
725/72

In [ ]:
result = model.evaluate(test_dataset)
y_true = np.argmax(np.concatenate([y.numpy() for _, y in test_dataset]))
y_pred = np.argmax(model.predict(test_dataset))

2025-03-17 18:06:53.807708: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:42: Filling up shuffle buffer (this may take a while): 29 of 128
2025-03-17 18:07:13.792388: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:42: Filling up shuffle buffer (this may take a while): 87 of 128
2025-03-17 18:07:18.572519: W tensorflow/core/kernels/data/cache_dataset_ops.cc:332] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2025-03-17 18:07:18.884170: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


104/104 ━━━━━━━━━━━━━━━━━━━━ 120s 810ms/step - accuracy: 0.5311 - loss: 0.6976 - precision: 0.0104 - recall: 0.2650


2025-03-17 18:08:53.667533: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


104/104 ━━━━━━━━━━━━━━━━━━━━ 100s 796ms/step


In [ ]:
report = classification_report(y_true, y_pred, target_names=LABELS)
print(report)

In [ ]:
conf_matrix = confusion_matrix(y_true, y_pred, normalize="true")
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, cmap="Blues", fmt=".2f", xticklabels=LABELS, yticklabels=LABELS)
plt.title("Confusion Matrix")
plt.show()

## SANDBOX

In [ ]:
train_metadata = pd.read_csv('/Users/mac/Development/diploma/skin-mutation-detection/data/train/ISIC_2019_Training_Metadata.csv')
test_metadata = pd.read_csv('/Users/mac/Development/diploma/skin-mutation-detection/data/test/ISIC_2019_Test_Metadata.csv')

In [ ]:
set(train_metadata.anatom_site_general.unique()) - set(test_metadata.anatom_site_general.unique())

{'lateral torso'}

In [ ]:
set(test_metadata.anatom_site_general.unique()) - set(train_metadata.anatom_site_general.unique())

set()

Summary: 
- test data doesn't contain lateral torso
- train metadata contains each class in test data

In [ ]:
recommended = ["MEL", "BCC", "SCC", "AK"]  # Needs doctor evaluation
ok_for_now = ["NV", "BKL", "DF", "VASC"]  # Likely harmless, monitor for changes

In [ ]:
train_groundtruth["category"] = train_groundtruth[recommended].sum(axis=1).apply(lambda x: "recommended" if x > 0 else "ok_for_now")

# Count occurrences of each category
counts = train_groundtruth["category"].value_counts()

In [ ]:
counts

category
ok_for_now     15991
recommended     9340
Name: count, dtype: int64

In [ ]:
base_model = EfficientNetV2M(weights="imagenet", include_top=True)
base_model.summary()

219330792/219330792 ━━━━━━━━━━━━━━━━━━━━ 23s 0us/step


Model: "efficientnetv2-m"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 480, 480,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_3         │ (None, 480, 480,  │          0 │ input_layer_6[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 240, 240,  │        648 │ rescaling_3[0][0] │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 240, 240,  │         96 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 240, 240,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 240, 240,  │      5,184 │ stem_activation[… │
│ (Conv2D)            │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 240, 240,  │         96 │ block1a_project_… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_ac… │ (None, 240, 240,  │          0 │ block1a_project_… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_add (Add)   │ (None, 240, 240,  │          0 │ block1a_project_… │
│                     │ 24)               │            │ stem_activation[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_co… │ (None, 240, 240,  │      5,184 │ block1a_add[0][0] │
│ (Conv2D)            │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_bn  │ (None, 240, 240,  │         96 │ block1b_project_… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_ac… │ (None, 240, 240,  │          0 │ block1b_project_… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_drop        │ (None, 240, 240,  │          0 │ block1b_project_… │
│ (Dropout)           │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_add (Add)   │ (None, 240, 240,  │          0 │ block1b_drop[0][… │
│                     │ 24)               │            │ block1a_add[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1c_project_co… │ (None, 240, 240,  │      5,184 │ block1b_add[0][0] │
│ (Conv2D)            │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1c_project_bn  │ (None, 240, 240,  │         96 │ block1c_project_… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1c_project_ac… │ (None, 240, 240,  │          0 │ block1c_project_

 Total params: 54,431,388 (207.64 MB)

 Trainable params: 54,139,356 (206.53 MB)

 Non-trainable params: 292,032 (1.11 MB)

In [ ]:
base_model = EfficientNetV2M(weights="imagenet", include_top=False)
base_model.summary()

Model: "efficientnetv2-m"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8       │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ None, 3)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_5         │ (None, None,      │          0 │ input_layer_8[0]… │
│ (Rescaling)         │ None, 3)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, None,      │        648 │ rescaling_5[0][0] │
│                     │ None, 24)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, None,      │         96 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ None, 24)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, None,      │          0 │ stem_bn[0][0]     │
│ (Activation)        │ None, 24)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, None,      │      5,184 │ stem_activation[… │
│ (Conv2D)            │ None, 24)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, None,      │         96 │ block1a_project_… │
│ (BatchNormalizatio… │ None, 24)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_ac… │ (None, None,      │          0 │ block1a_project_… │
│ (Activation)        │ None, 24)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_add (Add)   │ (None, None,      │          0 │ block1a_project_… │
│                     │ None, 24)         │            │ stem_activation[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_co… │ (None, None,      │      5,184 │ block1a_add[0][0] │
│ (Conv2D)            │ None, 24)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_bn  │ (None, None,      │         96 │ block1b_project_… │
│ (BatchNormalizatio… │ None, 24)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_ac… │ (None, None,      │          0 │ block1b_project_… │
│ (Activation)        │ None, 24)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_drop        │ (None, None,      │          0 │ block1b_project_… │
│ (Dropout)           │ None, 24)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_add (Add)   │ (None, None,      │          0 │ block1b_drop[0][… │
│                     │ None, 24)         │            │ block1a_add[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1c_project_co… │ (None, None,      │      5,184 │ block1b_add[0][0] │
│ (Conv2D)            │ None, 24)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1c_project_bn  │ (None, None,      │         96 │ block1c_project_… │
│ (BatchNormalizatio… │ None, 24)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1c_project_ac… │ (None, None,      │          0 │ block1c_project_

 Total params: 53,150,388 (202.75 MB)

 Trainable params: 52,858,356 (201.64 MB)

 Non-trainable params: 292,032 (1.11 MB)